In [ ]:
session.sql("""
DROP TABLE IF EXISTS tmp_person_role_assignments_stream_snapshot
""").collect()

In [ ]:
session.sql("""
CREATE TEMP TABLE tmp_person_role_assignments_stream_snapshot AS
SELECT *
FROM STREAM_T_PERSON_ROLE_ASSIGNMENTS
WHERE METADATA$ACTION='INSERT'
""").collect()

pra_raw = session.table("tmp_person_role_assignments_stream_snapshot")
print(f"snapshot person_role_assignments records found")

In [ ]:
pra_clean = pra_raw.select(
    col("PRA1_ID"),
    col("POI_POI_ID"),
    upper(trim(col("ROLE_TYPE"))).alias("ROLE_TYPE"),
    col("ADD_TS"),
    coalesce(trim(col("ADD_USER")), lit("SYSTEM")).alias("ADD_USER"),
    col("START_DATE"),
    col("END_DATE"),
    col("TICKLER_ID"),
    coalesce(trim(col("MOD_USER")), lit("SYSTEM")).alias("MOD_USER"),
    col("MOD_TS"),
    coalesce(trim(col("ROLE_SUB_TYPE")), lit("NA")).alias("ROLE_SUB_TYPE"),
    col("LOADED_DATE"),
    col("DELETED_DATE"),
    col("LOAD_TIMESTAMP").alias("RAW_LOAD_TIMESTAMP"),
    col("SOURCE_FILE_NAME")
)
print("pra clean")

In [ ]:
valid_pra = pra_clean.filter(
    col("PRA1_ID").is_not_null()
)

invalid_pra = pra_clean.filter(
    col("PRA1_ID").is_null()
).with_column(
    "ERROR_REASON",
    lit("PRA1_ID_NULL")
)
print("seperated valid and invalid records")

In [ ]:
pra_final = valid_pra.with_column(
"CHECKSUM",
sha2(
    concat_ws(
        lit("|"),
        coalesce(col("PRA1_ID").cast("string"), lit("")),
        coalesce(col("POI_POI_ID").cast("string"), lit("")),
        coalesce(col("ROLE_TYPE"), lit("")),
        coalesce(col("ROLE_SUB_TYPE"), lit("")),
        coalesce(col("ADD_USER"), lit("")),
        coalesce(col("MOD_USER"), lit("")),
        coalesce(col("START_DATE").cast("string"), lit("")),
        coalesce(col("END_DATE").cast("string"), lit("")),
        coalesce(col("TICKLER_ID").cast("string"), lit(""))
    ),
    256
)
)

In [ ]:
pra_final = pra_final.with_column(
    "STAGING_LOADED_TIMESTAMP",
    current_timestamp()
)

pra_final.write.save_as_table(
    table_name=f"{DB}.{STG}.{STG_PERSON_ROLE_ASSIGNMENTS}",
    mode="truncate"
)

print(f"PRA loaded into {STG}.{STG_PERSON_ROLE_ASSIGNMENTS}")

In [ ]:
invalid_pra.create_or_replace_temp_view("tmp_invalid_pra")

invalid_count = invalid_pra.count()

if invalid_count > 0:

    session.sql(f"""
    INSERT INTO {DB}.{STG}.{INVALID_RECORDS}
    (
        TABLE_NAME,
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        RAW_RECORD
    )
    SELECT
       'T_PERSON_ROLE_ASSIGNMENTS',
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        OBJECT_CONSTRUCT(*)
    FROM tmp_invalid_pra
    """).collect()
    print(f"Invalid records saved into {STG}.{INVALID_RECORDS}")
else:
    print("No invalid records")

In [ ]:
rows_processed = pra_final.count()
rows_failed = invalid_pra.count()

session.call(
   f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    session.sql("SELECT UUID_STRING()").collect()[0][0],
    'STG_PERSON_ROLE_ASSIGNMENTS_LOAD',
    'STAGING',
    datetime.now(),
    datetime.now(),
    rows_processed,
    rows_failed,
    'SUCCESS',
    None
)

session.call(
    f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    'SUCCESS',
    'STG_PERSON_ROLE_ASSIGNMENTS_LOAD',
    'STAGING',
    f'PERSON_ROLE_ASSIGNMENTS staging completed. Rows processed: {rows_processed}, Failed rows: {rows_failed}'
)
print("Audit log inserted and email sent")